In [2]:
import os
import random
import shutil
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from shutil import copyfile
import matplotlib.pyplot as plt
from tensorflow.keras.optimizers import RMSprop

In [3]:
source_path = 'data/tmp/PetImages'

current_dir = os.getcwd()

source_path_dogs = os.path.join(current_dir, "data/tmp/PetImages/Dog")
source_path_cats = os.path.join(current_dir,"data/tmp/PetImages/Cat")

print(f'There are {len(os.listdir(source_path_dogs))} images of dogs.')
print(f'There are {len(os.listdir(source_path_cats))} images of cats.')

There are 12501 images of dogs.
There are 12501 images of cats.


In [4]:
root_dir = 'data/tmp/cats-v-dogs'

# Empty directory to prevent FileExistsError is the function is run several times
if os.path.exists(root_dir):
    shutil.rmtree(root_dir)

# create_train_val_dirs
def create_train_val_dirs(root_dir):
    # Create Parent Directory
    os.mkdir(root_dir)

    for x in ["training", "validation"]:
        os.mkdir(root_dir + '/' + x)

        for y in ["cats", "dogs"]:
            os.mkdir(root_dir + '/' + x + '/' + y)

try:
    create_train_val_dirs(root_dir=root_dir)
except FileExistsError:
    print("You should not be seeing this since the upper directory is removed beforehand")

In [5]:
# Test your create_train_val_dirs function
for rootdir, dirs, files in os.walk(root_dir):
    for subdir in dirs:
        print(os.path.join(rootdir, subdir))

data/tmp/cats-v-dogs/training
data/tmp/cats-v-dogs/validation
data/tmp/cats-v-dogs/training/dogs
data/tmp/cats-v-dogs/training/cats
data/tmp/cats-v-dogs/validation/dogs
data/tmp/cats-v-dogs/validation/cats


In [6]:
# split_data
def split_data(SOURCE_DIR, TRAINING_DIR, VALIDATION_DIR, SPLIT_SIZE):
    """
    Splits the data into train and test sets

    Args:
        SOURCE_DIR (string): directory path containing the images
        TRAINING_DIR (string): directory path to be used for training
        VALIDATION_DIR (string): directory path to be used for validation
        SPLIT_SIZE (float): proportion of the dataset to be used for training

    Returns:
        None
    """

    files = []

    for filename in os.listdir(SOURCE_DIR):
        file = SOURCE_DIR + filename
        if os.path.getsize(file) > 0:
            files.append(file)
        else:
            print(f"{filename} is zero length, so ignoring.")

    training_length = int(len(files) * SPLIT_SIZE)
    testing_length = int(len(files) - training_length)
    shuffled_set = random.sample(files, len(files))
    training_set = shuffled_set[0:training_length]
    testing_set = shuffled_set[-testing_length:]
    
    for filename in training_set:
        this_file = SOURCE_DIR + os.path.basename(filename)
        destination = TRAINING_DIR + os.path.basename(filename)
        copyfile(this_file, destination)

    for filename in testing_set:
        this_file = SOURCE_DIR + os.path.basename(filename)
        destination = VALIDATION_DIR + os.path.basename(filename)
        copyfile(this_file, destination)

In [9]:
# Test split_data function


current_dir = os.getcwd()

# Define paths
CAT_SOURCE_DIR = os.path.join(current_dir,"data/tmp/PetImages/Cat/") 
DOG_SOURCE_DIR = os.path.join(current_dir,"data/tmp/PetImages/Dog/") 

TRAINING_DIR = "data/tmp/cats-v-dogs/training/"
VALIDATION_DIR = "data/tmp/cats-v-dogs/validation/"

TRAINING_CATS_DIR = os.path.join(current_dir, TRAINING_DIR, "cats/")
VALIDATION_CATS_DIR = os.path.join(current_dir, VALIDATION_DIR, "cats/")

TRAINING_DOGS_DIR = os.path.join(current_dir, TRAINING_DIR, "dogs/")
VALIDATION_DOGS_DIR = os.path.join(current_dir, VALIDATION_DIR, "dogs/")

# Empty directories in case you run this cell multiple times
if len(os.listdir(TRAINING_CATS_DIR)) > 0:
    for file in os.scandir(TRAINING_CATS_DIR):
        os.remove(file.path)

if len(os.listdir(TRAINING_DOGS_DIR)) > 0:
    for file in os.scandir(TRAINING_DOGS_DIR):
        os.remove(file.path)

if len(os.listdir(VALIDATION_CATS_DIR)) > 0:
    for file in os.scandir(VALIDATION_CATS_DIR):
        os.remove(file.path)

if len(os.listdir(VALIDATION_DOGS_DIR)) > 0:
    for file in os.scandir(VALIDATION_DOGS_DIR):
        os.remove(file.path)

# Define proportion of images used for training
split_size = .9

# Run the function
split_data(CAT_SOURCE_DIR,TRAINING_CATS_DIR,VALIDATION_CATS_DIR,split_size)
split_data(DOG_SOURCE_DIR,TRAINING_DOGS_DIR,VALIDATION_DOGS_DIR,split_size)

# Check that the number of images matches the expected output
print(f"\n\nOriginal cat's directory has {len(os.listdir(CAT_SOURCE_DIR))} images")
print(f"Original dogs's directory has {len(os.listdir(DOG_SOURCE_DIR))} images\n")

# Training and validation splits
print(f"There are {len(os.listdir(TRAINING_CATS_DIR))} images of cats for training")
print(f"There are {len(os.listdir(TRAINING_DOGS_DIR))} images of dogs for training")
print(f"There are {len(os.listdir(VALIDATION_CATS_DIR))} images of cats for validation")
print(f"There are {len(os.listdir(VALIDATION_DOGS_DIR))} images of dogs for validation")

666.jpg is zero length, so ignoring.
11702.jpg is zero length, so ignoring.


Original cat's directory has 12501 images
Original dogs's directory has 12501 images

There are 11250 images of cats for training
There are 11250 images of dogs for training
There are 1250 images of cats for validation
There are 1250 images of dogs for validation


In [10]:
def train_val_generators(TRAINING_DIR, VALIDATION_DIR):
    """
    Creates the training and validation data generators
    
    Args:
        TRAINING_DIR (string): directory path containing the training images
        VALIDATION_DIR (string): directory path containing the testing/validation images
        
    Returns:
        train_generator, validation_generator - tuple containing the generators
    """

    # Instantiate the ImageDataGenerator class 
    train_datagen = ImageDataGenerator(rescale=1.0/255.0)

    train_generator = train_datagen.flow_from_directory(
        directory = TRAINING_DIR,
        batch_size = 100,
        class_mode = 'binary',
        target_size = (150,150)
    )


    validation_datagen = ImageDataGenerator(rescale=1.0/255.0)

    validation_generator = validation_datagen.flow_from_directory(
        directory=VALIDATION_DIR,
        batch_size=100,
        class_mode='binary',
        target_size=(150,150)
    )

    return train_generator, validation_generator

In [11]:
# Test generators
train_generator, validation_generator = train_val_generators(TRAINING_DIR,VALIDATION_DIR)

Found 22498 images belonging to 2 classes.
Found 2500 images belonging to 2 classes.


In [18]:
def create_model():
    # DEFINE A KERAS MODEL TO CLASSIFY CATS V DOGS
    # USE AT LEAST 3 CONVOLUTION LAYERS

    model = tf.keras.models.Sequential([
        tf.keras.layers.Conv2D(16, (3,3), activation='relu', input_shape=(150, 150, 3)),
        tf.keras.layers.MaxPooling2D(2,2),
        tf.keras.layers.Conv2D(32, (3,3), activation='relu'),
        tf.keras.layers.MaxPooling2D(2,2), 
        tf.keras.layers.Conv2D(64, (3,3), activation='relu'), 
        tf.keras.layers.MaxPooling2D(2,2),
        # Flatten the results to feed into a DNN
        tf.keras.layers.Flatten(), 
        # 512 neuron hidden layer
        tf.keras.layers.Dense(512, activation='relu'), 
        # Only 1 output neuron. It will contain a value from 0-1 where 0 for 1 class ('cats') and 1 for the other ('dogs')
        tf.keras.layers.Dense(1, activation='sigmoid')  
    ])

    model.compile(
        optimizer=RMSprop(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    ) 

    return model

In [19]:
# Get the untrained model
model = create_model()

# Train the model
history = model.fit(train_generator,
                    epochs=15,
                    verbose=1,
                    validation_data=validation_generator)

Epoch 1/15


/Users/sohelshaikh/Development/ai/exploring_ai/.venv/lib/python3.11/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


 66/225 ━━━━━━━━━━━━━━━━━━━━ 55s 349ms/step - accuracy: 0.5078 - loss: 1.0026

/Users/sohelshaikh/Development/ai/exploring_ai/.venv/lib/python3.11/site-packages/PIL/TiffImagePlugin.py:890: UserWarning: Truncated File Read
  warnings.warn(str(msg))


225/225 ━━━━━━━━━━━━━━━━━━━━ 85s 371ms/step - accuracy: 0.5405 - loss: 0.8122 - val_accuracy: 0.7232 - val_loss: 0.5637
Epoch 2/15
225/225 ━━━━━━━━━━━━━━━━━━━━ 93s 411ms/step - accuracy: 0.7015 - loss: 0.5724 - val_accuracy: 0.7092 - val_loss: 0.5556
Epoch 3/15
225/225 ━━━━━━━━━━━━━━━━━━━━ 97s 427ms/step - accuracy: 0.7547 - loss: 0.4980 - val_accuracy: 0.7900 - val_loss: 0.4361
Epoch 4/15
225/225 ━━━━━━━━━━━━━━━━━━━━ 98s 430ms/step - accuracy: 0.7967 - loss: 0.4384 - val_accuracy: 0.8096 - val_loss: 0.4166
Epoch 5/15
225/225 ━━━━━━━━━━━━━━━━━━━━ 100s 440ms/step - accuracy: 0.8207 - loss: 0.3915 - val_accuracy: 0.7984 - val_loss: 0.4269
Epoch 6/15
225/225 ━━━━━━━━━━━━━━━━━━━━ 101s 446ms/step - accuracy: 0.8527 - loss: 0.3372 - val_accuracy: 0.8116 - val_loss: 0.4048
Epoch 7/15
225/225 ━━━━━━━━━━━━━━━━━━━━ 142s 627ms/step - accuracy: 0.8788 - loss: 0.2810 - val_accuracy: 0.8276 - val_loss: 0.3903
Epoch 8/15
225/225 ━━━━━━━━━━━━━━━━━━━━ 98s 432ms/step - accuracy: 0.9161 - loss: 0.2100 - 

In [ ]:
#-----------------------------------------------------------
# Retrieve a list of list results on training and test data
# sets for each training epoch
#-----------------------------------------------------------
acc=history.history['accuracy']
val_acc=history.history['val_accuracy']
loss=history.history['loss']
val_loss=history.history['val_loss']

epochs=range(len(acc)) # Get number of epochs

#------------------------------------------------
# Plot training and validation accuracy per epoch
#------------------------------------------------
plt.plot(epochs, acc, 'r', "Training Accuracy")
plt.plot(epochs, val_acc, 'b', "Validation Accuracy")
plt.title('Training and validation accuracy')
plt.show()
print("")

#------------------------------------------------
# Plot training and validation loss per epoch
#------------------------------------------------
plt.plot(epochs, loss, 'r', "Training Loss")
plt.plot(epochs, val_loss, 'b', "Validation Loss")
plt.show()